# Pipeline C: NOTEARS-Based Causal Discovery

This notebook implements NOTEARS for causal discovery on cross-sectional data.
- Implements linear NOTEARS algorithm for DAG discovery
- Uses continuous optimization with acyclicity constraint
- Produces directed acyclic graph (DAG)
- Outputs to timestamped artifact folder

## Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# Add utils to path
sys.path.insert(0, '/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/DAG discovery algorithms')
from causal_discovery_utils import *

# NOTEARS implementation
from scipy.optimize import minimize
from scipy.linalg import expm

print("✓ All imports successful")

## NOTEARS Implementation

In [ ]:
class NOTEARSLinear:
    """
    Linear NOTEARS algorithm for DAG discovery.
    """
    
    def __init__(self, lambda1=0.0, lambda2=0.0, rho_max=1e16, max_iter=100, h_tol=1e-8, w_threshold=None):
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.rho_max = rho_max
        self.max_iter = max_iter
        self.h_tol = h_tol
        self.w_threshold = w_threshold
        self.W_est = None
        
    def _h(self, W):
        """Compute acyclicity constraint h(W) = tr(exp(W ∘ W)) - d"""
        M = W * W
        E = expm(M)
        h = np.trace(E) - W.shape[0]
        return h
    
    def _func_nonlinear(self, W, X, rho):
        """Objective function with acyclicity constraint"""
        n, d = X.shape
        
        # Prediction error
        R = X - X @ W
        mse = np.sum(R ** 2) / n / d
        
        # Regularization
        reg = (self.lambda1 * np.sum(np.abs(W)) + 
               self.lambda2 * np.sum(W ** 2))
        
        # Acyclicity
        h = self._h(W)
        
        return mse + reg + rho * h + 0.5 * rho * h ** 2
    
    def fit(self, X):
        """Fit NOTEARS to data"""
        n, d = X.shape
        
        W = np.zeros((d, d))
        rho = 1.0
        h_prev = np.inf
        
        for iteration in range(self.max_iter):
            # Minimize objective
            result = minimize(
                lambda w: self._func_nonlinear(w.reshape(d, d), X, rho),
                W.flatten(),
                method='L-BFGS-B',
                options={'maxiter': 50}
            )
            
            W = result.x.reshape(d, d)
            h = self._h(W)
            
            if h < self.h_tol:
                break
            
            rho = min(rho * 1.5, self.rho_max)
            
            if abs(h - h_prev) < 1e-6:
                break
            
            h_prev = h
        
        # Apply threshold
        if self.w_threshold is not None:
            W[np.abs(W) < self.w_threshold] = 0
        
        self.W_est = W
        return W

print("✓ NOTEARSLinear class defined")

## Configuration

In [ ]:
# Pipeline configuration
PIPELINE_NAME = "NOTEARS_Based"
METRICS_CSV_PATH = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/Metrics data/metrics.csv"
ARTIFACTS_BASE = "/Users/manav.neema/Documents/Thesis/Experiment 2/Windows Setup/Causal Discovery/artifacts"

# Data parameters
MAX_RUNS_TO_PIVOT = 65

# NOTEARS parameters
LAMBDA1 = 0.0
LAMBDA2 = 0.0
RHO_MAX = 1e16
MAX_ITER = 100
H_TOL = 1e-8
W_THRESHOLD = None
W_PERCENTILE = 90

# Feature selection parameters
TARGET_FEATURES = 35
CORRELATION_THRESHOLD = 0.98

print(f"Pipeline C Configuration:")
print(f"  - Method: NOTEARS-Based")
print(f"  - Training runs: {MAX_RUNS_TO_PIVOT}")
print(f"  - Lambda1: {LAMBDA1}, Lambda2: {LAMBDA2}")
print(f"  - Max Iterations: {MAX_ITER}")
print(f"  - Weight Percentile: {W_PERCENTILE}")
print(f"  - Target Features: {TARGET_FEATURES}")
print(f"  - Metrics CSV: {METRICS_CSV_PATH}")

## Step 1: Load Metrics Data

In [ ]:
print("[Step 1] Loading metrics data...")
metrics_df = load_metrics_csv(METRICS_CSV_PATH)
print(f"Loaded {metrics_df.shape[0]} metric records")
print(f"Date range: {metrics_df['date'].min()} to {metrics_df['date'].max()}")

## Step 2: Transform Metrics to Matrix

In [ ]:
print("[Step 2] Transforming metrics to matrix...")
metrics_matrix = metrics_to_matrix(metrics_df, max_runs=MAX_RUNS_TO_PIVOT, stage=None)
print(f"Matrix shape: {metrics_matrix.shape}")

## Step 3: Preprocess Metrics Matrix

In [ ]:
print("\n[Step 3] Preprocessing metrics matrix...")
scaled, preprocess_meta = preprocess_metrics_matrix(
    metrics_matrix,
    zscore=True,
    feature_sample_ratio=3.0
)
print(f"After preprocessing: {scaled.shape}")

## Step 4: Feature Selection

In [ ]:
print("\n[Step 4] Feature selection...")
final_features, feature_selection_log = sophisticated_feature_selection(
    scaled,
    target_features=TARGET_FEATURES,
    variance_threshold=1e-6,
    correlation_threshold=CORRELATION_THRESHOLD
)

n_samples, n_features = final_features.shape
print(f"\nNOTEARS Requirements Check:")
print(f"  Samples: {n_samples}, Features: {n_features}")
print(f"  Sample-to-feature ratio: {n_samples / n_features:.2f}")

if final_features.isna().any().any():
    print("WARNING: NaN values still present!")

## Step 5: Generate Blacklist

In [ ]:
print("\n[Step 5] Generating blacklist...")
metric_cols = final_features.columns.tolist()
HUMAN_PRIOR_BLACKLIST = generate_stage_blacklist(metric_cols)
print(f"  - Blacklist edges: {len(HUMAN_PRIOR_BLACKLIST)}")

## Step 6: Run NOTEARS

In [ ]:
print(f"\n[Step 6] Running NOTEARS...")
print(f"  Lambda1: {LAMBDA1}, Lambda2: {LAMBDA2}")
print(f"  Max iterations: {MAX_ITER}")

try:
    model = NOTEARSLinear(
        lambda1=LAMBDA1,
        lambda2=LAMBDA2,
        max_iter=MAX_ITER,
        h_tol=H_TOL,
        rho_max=RHO_MAX,
        w_threshold=W_THRESHOLD
    )
    W_est = model.fit(final_features.values)
    print(f"  ✓ NOTEARS completed")
    print(f"  Learned weight matrix shape: {W_est.shape}")
except Exception as e:
    print(f"  ERROR: NOTEARS failed: {e}")
    W_est = None
    model = None

## Step 7: Extract Edges from Weight Matrix

In [ ]:
print(f"\n[Step 7] Extracting edges from weight matrix...")

raw_notears_edges = []
if W_est is not None:
    col_names = final_features.columns.tolist()
    
    for i in range(len(col_names)):
        for j in range(len(col_names)):
            if i != j and W_est[j, i] != 0:  # Note: W[j,i] is edge i->j
                from_col = col_names[i]
                to_col = col_names[j]
                weight = W_est[j, i]
                
                raw_notears_edges.append({
                    'from': from_col,
                    'to': to_col,
                    'edge_type': 'directed',
                    'weight': float(weight),
                    'abs_weight': float(abs(weight)),
                    'source': 'notears_raw'
                })

# Apply percentile threshold if specified
if raw_notears_edges and W_PERCENTILE is not None:
    weights = [e['abs_weight'] for e in raw_notears_edges]
    threshold = np.percentile(weights, W_PERCENTILE)
    raw_notears_edges = [e for e in raw_notears_edges if e['abs_weight'] >= threshold]

print(f"NOTEARS discovered {len(raw_notears_edges)} edges")
if len(raw_notears_edges) == 0:
    print("WARNING: No edges discovered by NOTEARS!")

## Step 8: Apply Blacklist Filtering

In [ ]:
print(f"\n[Step 8] Applying blacklist filtering...")

selected_feature_set = set(final_features.columns)
filtered_blacklist = [
    (a, b) for a, b in HUMAN_PRIOR_BLACKLIST 
    if a in selected_feature_set and b in selected_feature_set
]
blacklist_set = set(filtered_blacklist)

print(f"Applicable blacklist edges: {len(filtered_blacklist)}")

# Filter edges
filtered_edges = []
removed_by_blacklist = []

for edge in raw_notears_edges:
    edge_tuple = (edge['from'], edge['to'])
    if edge_tuple not in blacklist_set:
        filtered_edges.append(edge)
    else:
        removed_by_blacklist.append(edge)

print(f"After blacklist: {len(filtered_edges)} edges kept, {len(removed_by_blacklist)} removed")

## Step 9: Visualize Raw DAG

In [ ]:
print(f"\n[Step 9] Visualizing raw DAG...")
if len(raw_notears_edges) > 0:
    G_raw = visualize_dag(
        raw_notears_edges,
        title="Pipeline C: NOTEARS RAW DAG"
    )
    print(f"Raw DAG nodes: {G_raw.number_of_nodes()}, edges: {G_raw.number_of_edges()}")
else:
    print("No raw edges to visualize")

## Step 10: Visualize Filtered DAG

In [ ]:
print(f"\n[Step 10] Visualizing filtered DAG...")
if len(filtered_edges) > 0:
    G = visualize_dag(
        filtered_edges,
        title="Pipeline C: NOTEARS DAG (After Blacklist/Whitelist)"
    )
    print(f"Filtered DAG nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
else:
    print("No filtered edges to visualize")

## Step 11: Compute Baseline Statistics

In [ ]:
print(f"\n[Step 11] Computing baseline statistics...")
baseline_stats = compute_baseline_stats(final_features)
print(f"Computed baseline statistics for {len(baseline_stats)} metrics")

## Step 12: Build Adjacency Maps

In [ ]:
print(f"\n[Step 12] Building adjacency maps...")
upstream_map, downstream_map = build_adjacency_maps(filtered_edges, handle_undirected=False)
print(f"Upstream map: {len(upstream_map)} nodes")
print(f"Downstream map: {len(downstream_map)} nodes")

## Step 13: Export Artifacts

In [ ]:
print(f"\n[Step 13] Exporting artifacts...")

# Create timestamped artifact directory
pipeline_path = create_timestamped_artifact_dir(ARTIFACTS_BASE, PIPELINE_NAME)

print(f"\nSaving artifacts:")

# Save core artifacts
artifacts = {
    "pipeline": PIPELINE_NAME,
    "method": "notears-based",
    "data_type": "cross-sectional",
    "status": "SUCCESS" if W_est is not None else "FAILED",
    "timestamp": datetime.now().isoformat(),
    "training_runs": MAX_RUNS_TO_PIVOT,
    "preprocess_meta": preprocess_meta,
    "feature_selection_log": feature_selection_log,
    "notears_result": {
        "method": "notears",
        "lambda1": LAMBDA1,
        "lambda2": LAMBDA2,
        "w_threshold": W_THRESHOLD,
        "max_iter": MAX_ITER,
        "n_samples": n_samples,
        "n_features": n_features,
        "sample_feature_ratio": n_samples / n_features,
        "raw_edges_found": len(raw_notears_edges)
    },
    "edge_stats": {
        "raw_edges": len(raw_notears_edges),
        "removed_by_blacklist": len(removed_by_blacklist),
        "final_edges": len(filtered_edges),
        "edge_type": "directed",
        "orientation_method": "notears_optimization"
    },
    "raw_notears_edges": raw_notears_edges,
    "filtered_edges": filtered_edges,
    "blacklist_filtering": {
        "blacklist_edges_applicable": len(filtered_blacklist),
        "edges_removed": [(e['from'], e['to']) for e in removed_by_blacklist]
    },
    "final_graph_stats": {
        "total_edges": len(filtered_edges),
        "nodes_with_parents": len(upstream_map),
        "nodes_with_children": len(downstream_map),
        "is_dag": True
    }
}

# Save JSON artifacts
save_json_artifact(pipeline_path, 'causal_artifacts.json', artifacts)
save_json_artifact(pipeline_path, 'baseline_stats.json', baseline_stats)
save_json_artifact(pipeline_path, 'upstream_map.json', upstream_map)
save_json_artifact(pipeline_path, 'downstream_map.json', downstream_map)

# Save CSV artifacts
save_csv_artifact(pipeline_path, 'causal_metrics_matrix.csv', final_features)

if len(raw_notears_edges) > 0:
    raw_edges_df = pd.DataFrame(raw_notears_edges)
    raw_edges_df = raw_edges_df.sort_values('abs_weight', ascending=False)
    save_csv_artifact(pipeline_path, 'notears_raw_edges.csv', raw_edges_df)
    raw_upstream, raw_downstream = build_adjacency_maps(raw_notears_edges, handle_undirected=False)
    save_json_artifact(pipeline_path, 'raw_upstream_map.json', raw_upstream)
    save_json_artifact(pipeline_path, 'raw_downstream_map.json', raw_downstream)

if len(filtered_edges) > 0:
    filtered_edges_df = pd.DataFrame(filtered_edges)
    filtered_edges_df = filtered_edges_df.sort_values('abs_weight', ascending=False)
    save_csv_artifact(pipeline_path, 'notears_causal_edges.csv', filtered_edges_df)

print(f"\n" + "="*80)
print(f"✓ PIPELINE C COMPLETE — All artifacts saved")
print(f"="*80)
print(f"\nSaved to: {pipeline_path}")
print(f"\nFinal Results:")
print(f"  - RAW edges: {len(raw_notears_edges)}")
print(f"  - FILTERED edges: {len(filtered_edges)}")
print(f"  - Removed: {len(removed_by_blacklist)}")